<a href="https://colab.research.google.com/github/hpedrazamartinez/ENCH680-CO2-Adsorption-ML/blob/main/ENCH680_CO2_Adsorption_Reproduction_Henry_and_Kyra.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ENCH 680 Course Project: Reproducing Yuan et al. (2021)
**Applied Machine Learning for Prediction of CO₂ Adsorption on Biomass Waste-Derived Porous Carbons**
*Environ. Sci. Technol.* 2021, 55, 11925–11936. https://doi.org/10.1021/acs.est.1c01849

Group members: Kyra Kjargaard, Henry Pedraza Martinez. University of Calgary, ENCH 680.

**What this notebook does**
1. Loads the authors' dataset (527 data points).
2. Repeats the preprocessing: random-forest imputation of TPV and MPV from SA, then drops S (%).
3. Trains the Chapter 3 baselines (a single decision tree with and without pre-pruning, and a random forest).
4. Reproduces the paper's three boosted-tree models (GBDT, XGBoost, LightGBM) with an 85/15 split, 5-fold CV and grid search.
5. Compares our results with Table 1 of the paper.
6. Interprets the model with permutation importance (MDA) and partial dependence plots (PDPs).
7. Repeats the analysis on the RPC and HDPC sub-datasets.
8. Tests improvements: split sensitivity, leakage-safe grouped CV, SHAP, and the extrapolation limits of trees.

## 1. Data Loading

In [ ]:
raw = pd.read_excel("data.xlsx")
print(raw.shape)
display(raw.isna().sum().to_frame("missing values"))
raw.describe().T.round(2)

## 0. Install packages
Required packages: `pip install scikit-learn xgboost lightgbm shap pandas matplotlib seaborn openpyxl`

In [3]:
%pip install -q xgboost lightgbm shap openpyxl

In [2]:
import warnings, json, os
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import (train_test_split, GridSearchCV, cross_val_score, KFold, GroupKFold, cross_validate)
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import sklearn, xgboost, lightgbm

SEED = 42
os.makedirs("figs", exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.spines.top": False, "axes.spines.right": False})
print("scikit-learn", sklearn.__version__, "| xgboost", xgboost.__version__, "| lightgbm", lightgbm.__version__)
RESULTS = {}

scikit-learn 1.6.1 | xgboost 3.4.1 | lightgbm 4.6.0
